In [15]:
import tkinter as tk
from tkinter import filedialog, Label
from PIL import Image, ImageTk
import cv2
import torch
import torchvision
from torchvision.transforms import functional as F
import threading
import numpy as np

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=False)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, 21)
model.load_state_dict(torch.load("objectdetection_model.pth", map_location=device))
model.eval().to(device)

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=1e-05)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=1e-05)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=1e-05)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=1e-05)
          (relu

In [19]:
CLASSES = ['__background__', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car',
           'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person',
           'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor']

stop_flag = False
video_thread = None

def detect_objects(frame):
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img_tensor = F.to_tensor(img_rgb).to(device)

    with torch.no_grad():
        pred = model([img_tensor])[0]

    boxes = pred['boxes']
    scores = pred['scores']
    labels = pred['labels']
    threshold = 0.7
    keep = scores > threshold
    boxes, scores, labels = boxes[keep], scores[keep], labels[keep]

    for box, label, score in zip(boxes, labels, scores):
        box = box.cpu().numpy().astype(int)
        label_name = CLASSES[label.item()]
        cv2.rectangle(frame, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 2)
        cv2.putText(frame, f"{label_name}: {score:.2f}", (box[0], box[1]-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    return frame

def show_image(img):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_rgb)
    img_tk = ImageTk.PhotoImage(image=img_pil)
    video_label.config(image=img_tk)
    video_label.image = img_tk


In [21]:
def upload_video():
    global stop_flag
    stop_flag = False
    path = filedialog.askopenfilename()
    if not path:
        return
    if path.lower().endswith((".jpg", ".jpeg", ".png")):
        img = cv2.imread(path)
        detected = detect_objects(img)
        show_image(detected)
        result_label.config(text="Image Processed.")
    elif path.lower().endswith((".mp4", ".avi", ".mov")):
        run_video(path)

def run_video(video_path):
    def video_loop():
        global stop_flag
        cap = cv2.VideoCapture(video_path)
        while cap.isOpened() and not stop_flag:
            ret, frame = cap.read()
            if not ret:
                break
            detected = detect_objects(frame)
            show_image(detected)
            cv2.waitKey(1)
        cap.release()
        result_label.config(text="Video Finished.")

    global video_thread
    video_thread = threading.Thread(target=video_loop)
    video_thread.start()


In [29]:
def realtime():
    def webcam_loop():
        global stop_flag
        cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
        if not cap.isOpened():
            result_label.config(text="Error: Cannot access webcam.")
            return

        while not stop_flag:
            ret, frame = cap.read()
            if not ret:
                break
            detected = detect_objects(frame)
            show_image(detected)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        cap.release()
        result_label.config(text="Stopped Real-Time Video.")

    global video_thread, stop_flag
    stop_flag = False
    video_thread = threading.Thread(target=webcam_loop)
    video_thread.start()


def stop_video():
    global stop_flag
    stop_flag = True

def reset():
    global stop_flag
    stop_flag = True
    video_label.config(image="")
    result_label.config(text="")

In [33]:
root = tk.Tk()
root.title("Object Detection")
root.geometry("800x750")

heading = Label(root, text="Object Detection", font=("Times New Roman", 24, "bold"),
                justify="center", fg="dark green")
heading.pack(pady=10)

btn_realtime = tk.Button(root, text="Real-Time Video", command=realtime, width=30, height=3)
btn_realtime.pack(pady=10)

btn_upload = tk.Button(root, text="Upload Video/Image", command=upload_video, width=30, height=3)
btn_upload.pack(pady=10)

btn_frame = tk.Frame(root)
btn_frame.pack(pady=10)

btn_stop = tk.Button(btn_frame, text="Stop", command=stop_video, width=15, height=2)
btn_stop.pack(side="left", padx=5)

btn_reset = tk.Button(btn_frame, text="Reset", command=reset, width=15, height=2)
btn_reset.pack(side="left", padx=5)

video_label = tk.Label(root)
video_label.pack()

result_label = tk.Label(root, text="", font=("Times New Roman", 16))
result_label.pack(pady=10)

root.mainloop()